3D 이미지로 CT는 흑백 이미지처럼 하나의 밀도 채널만 있다. 즉 다른 색상 채널은 그대로 둔 채 데이터를 저장하기도 한다.

2차원 단면을 스택처럼 쌓아 3차원 텐서로 만들면 대상의 3차원 해부도를 표현한 용적 데이터를 만들 수 있다.

용적 데이터를 저장하는 텐서와 이미지 데이터는 근본적으로 큰 차이가 없다는 정도,

채널 차원 뒤에 부가적으로 깊이 차원을 가지기 때문에 `N x C x D x H x W` 5차원 텐서가 된다.

## 4.2.1 특수 포맷 로딩
imageio 모듈에 있는 volread 함수 사용
- 의료용 디지털 영상(DICOM) 파일을 연속된 형태로 조합해 3차원 넘파이 배열 생성

In [1]:
import numpy as np
import torch
torch.set_printoptions(edgeitems=2, threshold=50)

In [2]:
import imageio

dir_path = '/content/2-LUNG 3.0  B70f-04083'
vol_arr = imageio.volread(dir_path, 'DICOM')
vol_arr.shape

Reading DICOM (examining files): 1/99 files (1.0%)99/99 files (100.0%)
  Found 1 correct series.
Reading DICOM (loading data): 99/99  (100.0%)


(99, 512, 512)

In [3]:
# unsqueeze 를 사용해서 channel 차원을 위한 공간 생성
vol = torch.from_numpy(vol_arr).float()
vol = torch.unsqueeze(vol, 0)

vol.shape

torch.Size([1, 99, 512, 512])

batch 방향을 따라 여러 용적 데이터를 쌓으면 5차원 데이터셋을 만들 수 있다.

# 4.3 테이블 데이터 표현하기
테이블 형식의 데이터(CSV)는 일반적으로 각 열의 타입이 동일하지 않다.

즉 열이 다르면 데이터 타입도 다르기에 어떤 열에서는 사과의 무게를, 다른 열에서는 사과의 색상을 인코딩해 레이블에 담을 수 있다.

반면 파이토치 텐서의 내부 값들은 형태가 동일(숫자로 인코딩하기에 정수나 부울 값을 가질 수도 있지만 대부분 부동소수점 값을 가짐)

신경망 자체가 실수값을 입력으로 받아 연속적인 행렬 곱셈이나 비선형 함수 같은 연속적인 연산을 거치며 실수 값을 출력으로 만들기 때문

## 4.3.1 실세계 데이터셋 사용하기
- 와인 데이터셋 사용
- 12개 콤마로 구분된 값 나열, 처음 11개 열은 화학적 성분 값, 마지막은 맛을 점수로 나타냄(0~10)
- 머신러닝 작업은 화학적 성분만 보고 맛을 예측.
- 화학적 성분 열과 품질 열과의 상관관계를 찾기(황 성분이 줄어들수록 맛이 좋아지는 관계)

## 4.3.2 와인 데이터를 텐서로 읽어오기
파이썬으로 데이터를 읽은 후 어떻게 파이토치 텐서로 바꿀까?
- 파이썬에 내장된 csv 모듈을 사용하는 법
- 넘파이
- 판다스
파일을 읽고, 만들어진 넘파이 배열을 파이토치 텐서로 바꾸기

In [4]:
import csv
wine_path = '/content/winequality-white.csv'
wineq_numpy = np.loadtxt(wine_path, dtype=np.float32, delimiter=';', skiprows=1) #skiprows는 첫행 생략

wineq_numpy

array([[ 7.  ,  0.27,  0.36, ...,  0.45,  8.8 ,  6.  ],
       [ 6.3 ,  0.3 ,  0.34, ...,  0.49,  9.5 ,  6.  ],
       [ 8.1 ,  0.28,  0.4 , ...,  0.44, 10.1 ,  6.  ],
       ...,
       [ 6.5 ,  0.24,  0.19, ...,  0.46,  9.4 ,  6.  ],
       [ 5.5 ,  0.29,  0.3 , ...,  0.38, 12.8 ,  7.  ],
       [ 6.  ,  0.21,  0.38, ...,  0.32, 11.8 ,  6.  ]], dtype=float32)

In [5]:
# 각주를 리스트의 형태로 반환(;)<- 첫번째 요소
# next는 클래스 순서 생성
col_list = next(csv.reader(open(wine_path), delimiter=';'))

wineq_numpy.shape, col_list

((4898, 12),
 ['fixed acidity',
  'volatile acidity',
  'citric acid',
  'residual sugar',
  'chlorides',
  'free sulfur dioxide',
  'total sulfur dioxide',
  'density',
  'pH',
  'sulphates',
  'alcohol',
  'quality'])

In [6]:
# 넘파이 배열을 파이토치 텐서로 변환
wineq = torch.from_numpy(wineq_numpy)
wineq.shape, wineq.dtype

(torch.Size([4898, 12]), torch.float32)

이제 맛 점수가 기록된 마지막 열까지 들어간 부동소수점 torch.Tensor을 만들었다

## 4.3.3 점수 표현하기
- 품질 점수를 연속값으로 취급, 실수 형태를 유지한 채 회귀 작업 수행
- 레이블로 취급, 분류 작업에서 화학 정보를 분석해 레이블을 예측하는 시도
- 두 경우 모두, 입력 데이터의 텐서에게 점수를 제거해 독자적인 텐서로 유지, 모델에 입력 하는 대신 점수를 정답 값으로 사용 가능

In [7]:
data = wineq[:, :-1]
data, data.shape

(tensor([[ 7.0000,  0.2700,  ...,  0.4500,  8.8000],
         [ 6.3000,  0.3000,  ...,  0.4900,  9.5000],
         ...,
         [ 5.5000,  0.2900,  ...,  0.3800, 12.8000],
         [ 6.0000,  0.2100,  ...,  0.3200, 11.8000]]),
 torch.Size([4898, 11]))

In [8]:
target = wineq[:, -1] # 모든 행과 마지막 열을 선택
target, target.shape

(tensor([6., 6.,  ..., 7., 6.]), torch.Size([4898]))

레이블 텐서의 target 텐서를 전치할 수 있는 두 가지 방법
1. 점수를 담은 정수 벡터로 레이블을 처리
2. 원핫 인코딩()

1번일 때는 와인의 품질 점수를 처리할 때 점수상에서 순서가 있음을 가정하여 점수 간 거리도 가정할 때(점수 1과 3의 차이는 점수 2와 4의 차이와 동일)

2번일 때는 점수가 포도 품종처럼 서로 간에 완전히 이산적인 경우라면 값 사이의 순서나 거리 개념이 없을 때(2.4 처럼 정수 점수 사이의 분수 값은 의미가 없고 양적인 점수를 따지는 경우에 적합)

In [9]:
# 1. 정수 벡터로 레이블 처리
target = wineq[:, -1].long()  # long()은 int형으로 처리하기 위해
target

tensor([6, 6,  ..., 7, 6])

'와인 색' 처럼 값이 문자열로 이루어진 테이블이라면 각 문자열마다 대응하는 정수를 할당해서 같은 식으로 처리할 수 있다.

## 4.3.4 원핫 인코딩
1부터 10까지의 값이 벡터 안의 10개의 원소에 대응하도록 정해두고서 원소 하나만 1로 설정하고 나머지는 모두 0으로 설정하는 방법

예)
값 1: 벡터(1,0,0,0,0,0,0,0,0,0)
값 5: 벡터(0,0,0,0,1,0,0,0,0,0) 로 나타냄

점수를 인코딩하면서 공교롭게도 1이 나타난 순서가 점수와 일치했는데 분류 관점에서 보면 다른 순서로 정해도 상관 없다.



In [10]:
# 원핫 인코딩은 scatter_ 메소드를 사용하여 소스 텐서와 함께 전달된 인자의 인덱스를 따라 새 텐서를 채워 넘겨준다
target_onehot = torch.zeros(target.shape[0], 10) # zeros로 raw의 개수(shape[0])은 10개로 채운다는 뜻
target_onehot.scatter_(1, target.unsqueeze(1), 1.0) # 벡터(스칼라 형태로 된 레이블을 담긴 구조)를 행렬화 하기 위해

tensor([[0., 0.,  ..., 0., 0.],
        [0., 0.,  ..., 0., 0.],
        ...,
        [0., 0.,  ..., 0., 0.],
        [0., 0.,  ..., 0., 0.]])

### `scatter_` 메소드
_는 텐서를 바꿔치기 하는 방법으로 변경하는 메소드
- 뒤에 오는 두개의 인자가 따라야 하는 차원 명세(dim)
- 원핫으로 인코딩할 요소를 가리키는 인덱스가 들어있는 텐서(index)
- 원핫 인코딩할 원소가 들어있는 텐서 혹은 단일 스칼라(value)

In [11]:
# index 텐서는 텐서를 원핫인코딩할 때 동일한 차원 수를 만들기 위해 필요.
# target_onehot이 2차원이므로 `unsqueeze`를 사용해 target에 추가 차원을 만들어준다
target_unsqueezed = target.unsqueeze(1)
target_unsqueezed

tensor([[6],
        [6],
        ...,
        [7],
        [6]])

## 4.3.5 언제 카테고리화 할 것인가?
연속 데이터, 순서 데이터, 카테고리 데이터로 열을 다루는 방법

```
**열 데이터**
|
`연속 데이터인가?`    -(네)->   `값을 바로 사용`
|                           |(연속으로 취급)
(아니요)                     (네)
|                           |
`순서 데이터인가?`    -(네)->   `우선 순위가 있는 순서인가?`
|                           |
(아니요)                     (아니요)
|                           |(카테고리로 취급)
`카테고리 데이터인가?` -(네)->   `원핫이나 임베딩을 사용한다`
```

In [12]:
# 화학 분석값 11개 변수 포함된 data 텐서, API 함수로 텐서 형태의 데이터 가공
data_mean = torch.mean(data, dim=0) # 각 열의 평균과 표준편차 구하기
data_mean

tensor([6.8548e+00, 2.7824e-01, 3.3419e-01, 6.3914e+00, 4.5772e-02, 3.5308e+01,
        1.3836e+02, 9.9403e-01, 3.1883e+00, 4.8985e-01, 1.0514e+01])

In [13]:
data_var = torch.var(data, dim=0) # 각 열의 평균과 표준편차 구하기
data_var

tensor([7.1211e-01, 1.0160e-02, 1.4646e-02, 2.5726e+01, 4.7733e-04, 2.8924e+02,
        1.8061e+03, 8.9455e-06, 2.2801e-02, 1.3025e-02, 1.5144e+00])

### dim=0 은 가장 바깥차원(행 방향으로 처리)
- [1,2][3,4] -> [1+3,2+4]
### dim=1 은 안쪽 차원(열 방향으로 처리)
- [1,2][3,4] -> [1+2,3+4]

이와 같은 연산은 축소(reduction)연산 이 수행

In [14]:
data_normalizated = (data - data_mean) / torch.sqrt(data_var) # 열의 값에서 평균을 뺴고 표준편차로 나눠 정규화 진행
data_normalizated

tensor([[ 1.7208e-01, -8.1761e-02,  ..., -3.4915e-01, -1.3930e+00],
        [-6.5743e-01,  2.1587e-01,  ...,  1.3422e-03, -8.2419e-01],
        ...,
        [-1.6054e+00,  1.1666e-01,  ..., -9.6251e-01,  1.8574e+00],
        [-1.0129e+00, -6.7703e-01,  ..., -1.4882e+00,  1.0448e+00]])

## 4.3.6 임계값으로 찾기
눈으로 들여다보고 좋은 와인인지 나쁜 와인인지 구별할 쉬운 방법

In [15]:
# 점수 3 이하인 열을 target에서 걸러내기
bad_indexes = target <= 3
bad_indexes.shape, bad_indexes.dtype, bad_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(20))

### 고급 인덱싱(Advanced indexing)
data 텐서를 인덱싱할 때 torch.bool 데이터 타입을 사용하면 열이 True에 해당하는 행들만 접근할 수 있다.

텐서에 들어있는 요소에 대해 임계값으로 비교하여 False와 True에 따라 골라낸 상태

In [16]:
bad_data = data[bad_indexes]
bad_data.shape

torch.Size([20, 11])

In [17]:
# 새 bad_data 텐서에는 20개 행, bad_indexes 텐서에서 True 값 행의 수와 동일.
# 다른 11개의 열은 그대로이므로 .mean() 값을 뽑아보자.
bad_data = data[target <= 3]
mid_data = data[(target > 3) & (target < 7)]  # & 연산자를 통한 부울 넘파이 배열과 파이토치 텐서의 논리 and 연산
good_data = data[target >= 7]

bad_mean = torch.mean(bad_data, dim=0)
mid_mean = torch.mean(mid_data, dim=0)
good_mean = torch.mean(good_data, dim=0)

for i, args in enumerate(zip(col_list, bad_mean, mid_mean, good_mean)):
  print('{:2} {:20} {:6.2f} {:6.2f} {:6.2f}'.format(i, *args))

 0 fixed acidity          7.60   6.89   6.73
 1 volatile acidity       0.33   0.28   0.27
 2 citric acid            0.34   0.34   0.33
 3 residual sugar         6.39   6.71   5.26
 4 chlorides              0.05   0.05   0.04
 5 free sulfur dioxide   53.33  35.42  34.55
 6 total sulfur dioxide 170.60 141.83 125.25
 7 density                0.99   0.99   0.99
 8 pH                     3.19   3.18   3.22
 9 sulphates              0.47   0.49   0.50
10 alcohol               10.34  10.26  11.42


### 결과
나쁜 와인은 이산화황 성분이 다른 경우보다 높다.

와인의 평가 기준으로 이산화황 총량을 **임계값**으로 사용할 수 있다.

이산화황 총량에서 앞서 계산해 놓은 **중앙점보다 낮은 인덱스**만 가져와본다.

In [18]:
total_sulfur_threshold = 141.83
total_sulfur_data = data[:, 6]  # 중앙점 보다 낮게
predicted_indexes = torch.lt(total_sulfur_data, total_sulfur_threshold)

In [19]:
predicted_indexes.shape, predicted_indexes.dtype, predicted_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(2727))

In [20]:
# 전체 와인 중 절반 이상을 높은 품질로 구별할 수 있는 임계값이다.
actual_indexes = target > 5
actual_indexes.shape, actual_indexes.dtype, actual_indexes.sum()

(torch.Size([4898]), torch.bool, tensor(3258))

임계값으로 예측한 수량보다 좋은 와인은 500개나 더 많이 존재하므로 증거 확보.

실제 순위와 우리가 예측한 순서가 얼마나 잘 맞는지 확인 필요.

In [21]:
# 예측 순위와 실제 품질 순위에 and 연산 수행, 교집합으로 평가
n_matches = torch.sum(actual_indexes & predicted_indexes).item()
n_predicted = torch.sum(predicted_indexes).item()
n_actual = torch.sum(actual_indexes).item()

n_matches, n_matches / n_predicted, n_matches

(2018, 0.74000733406674, 2018)

대략 2000여개의 와인을 맞췄다. 2700개를 예측했으니 74% 확률로 고품질의 와인 분류.

실제로는 와인의 품질에 당연히 더 많은 변수들의 작용하고 이 변수들과 품질 사이에는 숫자 하나를 경계값으로 사용하는 것보다 훨씬 복잡한 관계가 존재.